# Endpoints

ESA :: https://earth.esa.int/eogateway/search?category=Data&data_type=data+description&sortby=RELEVANCE <br>
Sentinel-Sat :: https://sentinelsat.readthedocs.io/en/stable/ <br>
Sentinel-Hub :: https://apps.sentinel-hub.com/dashboard/#/account/settings <br>
Submit an account :: https://nor-discover.cloudeo.group/?textSearch=&filterServiceType=DPaaS&filterSource=Any&filterGeographicalCoverage=Any&filterTemporalPeriodStart=&filterTemporalPeriodEnd=


# OSM

## example about buildings in Napoli

In [ ]:
pip install pystac-client shapely

In [ ]:
pip install overpy

In [ ]:
pip install osmnx

In [ ]:
pip install pyproj rioxarray

In [ ]:
# import osmnx
import osmnx as ox
import geopandas as gpd

In [ ]:
#place_name = "Edgewood Washington, DC, USA"
place_name = "Naples, Italy"

In [ ]:
# Get place boundary related to the place name as a geodataframe
area = ox.geocode_to_gdf(place_name)

In [ ]:
# Check the data type
area

In [ ]:
type(area)

In [ ]:
area.plot()

In [ ]:
# List key-value pairs for tags
tags = {'building': True}   

buildings = ox.geometries_from_place(place_name, tags)

In [ ]:
#buildings.head()

In [ ]:
#buildings.tail()

In [ ]:
# Plot footprints 
buildings.plot()

In [ ]:
api_url = 'https://earth-search.aws.element84.com/v1'

In [ ]:
import pystac_client

In [ ]:
client = pystac_client.Client.open(api_url)

In [ ]:
for collection in client.get_collections():
  print(collection)

In [ ]:
collection = 'sentinel-2-l2a'

In [ ]:
datetime = '2023-04-01/2023-05-03'

In [ ]:
area.geometry[0]

In [ ]:
search = client.search(
    collections = [collection],
    intersects = area.geometry[0],
    datetime = datetime,
)

In [ ]:
search.matched()

In [ ]:
items = search.get_all_items()

In [ ]:
len(items)

In [ ]:
for item in items:
    print(item)

In [ ]:
item = items[0]

In [ ]:
item.geometry

In [ ]:
item.assets.keys()

In [ ]:
item.assets

In [ ]:
item.assets["green"].href

In [ ]:
asset = item.assets["green"]

In [ ]:
dir(asset)

In [ ]:
asset.extra_fields

In [ ]:
import rioxarray

In [ ]:
green = rioxarray.open_rasterio(items[0].assets["green"].href)

In [ ]:
green

In [ ]:
green.values

In [ ]:
green.plot()

In [ ]:
import requests
url = items[0].assets["green"].href
response = requests.get(url)
with open("image.jpg", "wb") as f:
    f.write(response.content)

## GEE openEO

In [ ]:
import pystac_client

In [ ]:
from shapely.geometry import Point # vers 1.8.5 / vers 2 is different

In [ ]:
pip show shapely

In [ ]:
endpoint = "https://earthengine.openeo.org/v1.0/"

In [ ]:
endpoint = "https://tamn.snapplanet.io"

In [ ]:
client = pystac_client.Client.open(endpoint)

In [ ]:
for collection in client.get_collections():
    print(collection)

In [ ]:
collection = "S2"

In [ ]:
search = client.search(
    collections=[collection],
    intersects=Point(13.5, 43.3),
    datetime='2023-04-01/2023-05-03',
    # query=["eo:cloud_cover<15"],
)

In [ ]:
search.matched()

In [ ]:
endpoint = "https://openeo.eodc.eu/v1.0/"

In [ ]:
client = pystac_client.Client.open(endpoint)

In [ ]:
for collection in client.get_collections():
    print(collection)

In [ ]:
collection = "Land"

In [ ]:
search = client.search(
    collections=[collection],
    intersects=Point(13.5, 43.3),
    datetime='2023-04-01/2023-05-03',
    # query=["eo:cloud_cover<15"],
)

In [ ]:
search.matched()

In [ ]:
endpoint = "https://catalogue.dataspace.copernicus.eu/stac/"

In [ ]:
pip show pystac-client

#### Batch OSM data :: psql insert

In [ ]:
from impervious.config import pg_url  # connessione da .env
db_connection_url = pg_url()
con = create_engine(db_connection_url)  

In [ ]:
sql = """
CREATE TABLE [IF NOT EXISTS] buildings (
   column1 VARCHAR(50) ,
   column2 VARCHAR(50) ,
   column3 VARCHAR(50)
)
"""
sql

In [ ]:
cities_gdf = gpd.read_postgis(sql, con)

In [ ]:
tmp = geopandas.read_file("osm_data/buildings__Lettere.geojson")

In [ ]:
tmp.head(2)

In [ ]:
#for col in tmp.columns:
#    print(col)
tmp.columns

In [ ]:
from impervious.config import pg_url  # connessione da .env
db_connection_url = pg_url()
con = create_engine(db_connection_url)

In [ ]:
# test connection
con.connect()

In [ ]:
tmp = geopandas.read_file("osm_data/buildings__Lettere.geojson")

In [ ]:
tmp.geometry.type.unique()

In [ ]:
print(tmp.shape)

In [ ]:
tmp = tmp[tmp.geometry.type=="Polygon"]

In [ ]:
tmp.head(3)

In [ ]:
from geoalchemy2 import Geometry, WKTElement
from sqlalchemy import *

In [ ]:
tmp['geometry'] = tmp['geometry'].apply(lambda x: WKTElement(x.wkt, srid=4326))

In [ ]:
tmp.to_sql("buildings", con, "public", if_exists='append', index=False, 
                         dtype={'geometry': Geometry('POLYGON', srid=4326)})

# Copernicus

https://dataspace.copernicus.eu/analyse/apis/catalog-apis
<br>
https://pystac-client.readthedocs.io/en/stable/quickstart.html
<br>
https://stacindex.org/catalogs/eodc-openeo#/5WpJuuvfexLDmorjk1ZqxCkv4xHai3qs8ty7AQd7?t=2

In [ ]:
from pystac_client import Client

In [ ]:
client = Client.open("https://earth-search.aws.element84.com/v0")

In [ ]:
client.title

In [ ]:
my_search = client.search(
    max_items=10,
    collections=['sentinel-s2-l2a-cogs'],
    bbox=[-72.5,40.5,-72,41])
print(f"{my_search.matched()} items found")

In [ ]:
for item in my_search.items():
    print(item.id)

# SentinelSat | Python

GitHub :: https://github.com/sentinelsat/sentinelsat <br>
Copernicus Open Access Hub :: https://scihub.copernicus.eu/dhus/#/home <br>

In [ ]:
user = "giulange"
pswd = "voxcak-gewkas-7tezSa"

In [ ]:
# connect to the API
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
from datetime import date

In [ ]:
api = SentinelAPI(user, pswd, 'https://apihub.copernicus.eu/apihub')

In [ ]:
api.api_url

## Query

In [ ]:
sa = "italy_center_south.geojson"

In [ ]:
# search by polygon, time, and SciHub query keywords
footprint = geojson_to_wkt(read_geojson(sa))
products = api.query(footprint,
                     date=( date(2023,4,1), date(2023, 5, 1) ),
                     platformname='Sentinel-2',
                     cloudcoverpercentage=(0, 30))

In [ ]:
len(products)

In [ ]:
type(products)

In [ ]:
key1 = products["00679a3f-2d4f-4416-bc95-986317533104"]["uuid"]

In [ ]:
products[key1]

In [ ]:
# Get basic information about the product: its title, file size, MD5 sum, date, footprint and
# its download url
api.get_product_odata(key1)

In [ ]:
# Get the product's full metadata available on the server
api.get_product_odata(key1, full=True)

In [ ]:
LIST = products.keys()

## Download

In [ ]:
# download all results from the search
api.download(key1)

## Load raster

In [ ]:
import rioxarray

In [ ]:
ras_path_20m = "data/S2A_MSIL2A_20230420T100021_N0509_R122_T33TVF_20230420T155453.SAFE/GRANULE/L2A_T33TVF_A040874_20230420T100737/IMG_DATA/R20m/"

In [ ]:
raster = rioxarray.open_rasterio(ras_path_20m + "T33TVF_20230420T100021_B02_20m.jp2")

In [ ]:
raster

In [ ]:
ras_multiband = rioxarray.open_rasterio(ras_path_20m + "T33TVF_20230420T100021_TCI_20m.jp2")

In [ ]:
ras_multiband.rio.crs

In [ ]:
ras_multiband

## Plot

In [ ]:
raster.plot(robust=True)

In [ ]:
ras_multiband.plot()

In [ ]:
ras_multiband.plot.imshow()

In [ ]:
ras_masked.values

## GeoDataFrame from products

In [ ]:
gdf = api.to_geodataframe(products)

In [ ]:
type(gdf)

In [ ]:
gdf.head()

In [ ]:
gdf.plot()

## Interactive map

In [ ]:
gdf.explore()

In [ ]:
import folium

In [ ]:
m = folium.Map([43.5, 12.5], zoom_start=11)
boundary = gpd.read_file(r'italy_center_south.geojson')
folium.GeoJson(boundary).add_to(m)
m

## LTA-Products
https://sentinelsat.readthedocs.io/en/stable/api_overview.html#lta-products <br>
Using the command `download_all()` it will download all online and offline products.

## Raster split / tiling

In [ ]:
import gdaltest

import gdaltest<br>
check here: https://github.com/OSGeo/gdal/blob/master/autotest/pymod/gdaltest.py

# SentinelHub-py

To create sentinelhub credentials:<br>
https://www.youtube.com/watch?v=CBIlTOl2po4&t=1760s

##### pip show sentinelhub

In [ ]:
pip install --upgrade pdpbox matplotlib requests-oauthlib pyproj utm click tqdm oauthlib aenum dataclasses-json  tifffile python-dateutil tomli tomli-w requests numpy pillow shapely typing-extensions

In [ ]:
pip install utils

In [ ]:
from sentinelhub import SHConfig

In [ ]:
config = SHConfig()

In [ ]:
config

In [ ]:
config.instance_id = '92da91b8-dd15-4d37-a6e6-95bea2ef0796'
config.sh_client_id = '0f25240f-ef48-4cc7-b529-a725789b39cc'
config.sh_client_secret = '<)AeK<}d#@Y-|w3LhU>]]<t9^TXBL@{CrpN,PC?0'

In [ ]:
config_name = "python-test-profile-giuliano"

In [ ]:
config.save( config_name )

## Prerequisites
https://sentinelhub-py.readthedocs.io/en/latest/examples/process_request.html#Imports

In [ ]:
from sentinelhub import SHConfig
config = SHConfig(config_name)

In [ ]:
config

In [ ]:
if not config.sh_client_id or not config.sh_client_secret:
    print("Warning! To use Process API, please provide the credentials (OAuth client ID and client secret).")

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import datetime
import os

import matplotlib.pyplot as plt
import numpy as np

from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    DownloadRequest,
    MimeType,
    MosaickingOrder,
    SentinelHubDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
)

# The following is not a package. It is a file utils.py which should be in the same folder as this notebook.
from utils import plot_image

In [ ]:
# longitude and latitude coordinates of lower left and upper right corners
coords_wgs84 = (12.30,41.70, 12.65,42.05)
#coords_wgs84 = (46.16, -16.15, 46.51, -15.58)

In [ ]:
resolution = 60
sa_bbox = BBox(bbox=coords_wgs84, crs=CRS.WGS84)
sa_size = bbox_to_dimensions(sa_bbox, resolution=resolution)

print(f"Image shape at {resolution} m resolution: {sa_size} pixels")

In [ ]:
evalscript_true_color = """
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04"]
            }],
            output: {
                bands: 3
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B04, sample.B03, sample.B02];
    }
"""

request_true_color = SentinelHubRequest(
    evalscript=evalscript_true_color,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L1C,
            time_interval=("2023-04-12", "2023-04-25"),
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
    bbox=sa_bbox,
    size=sa_size,
    config=config,
)

In [ ]:
true_color_imgs = request_true_color.get_data()

In [ ]:
print(f"Returned data is of type = {type(true_color_imgs)} and length {len(true_color_imgs)}.")
print(f"Single element in the list is of type {type(true_color_imgs[-1])} and has shape {true_color_imgs[-1].shape}")

In [ ]:
image = true_color_imgs[0]
print(f"Image type: {image.dtype}")

# plot function
# factor 1/255 to scale between 0-1
# factor 3.5 to increase brightness
plot_image(image, factor=3.5 / 255, clip_range=(0, 1))